In [ ]:
#Importo Pacchetti Necessari
import pandas as pd
import numpy as np
import os
import json
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.preprocessing import  MinMaxScaler, OneHotEncoder
from sklearn.metrics import log_loss,accuracy_score, roc_auc_score, silhouette_score
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV 
import requests


In [ ]:

# 1. CONFIGURATION
# ─────────────────────────────────────────────

OLLAMA_URL    = "http://localhost:11434/api/chat"
MODELLO       = "phi3:mini"
MAGIC_WORD    = "yes"

CONTEXT_MEMORY_FILE = "context_memory.json"
OUTPUT_JSON         = "profile_descriptions.json"
TIMEOUT             = 5000

# All possible column names that identify a profile
# (case-sensitive — add more here if needed)
PROFILE_COLUMNS = ["profile_ID", "Profile_ID", "profile_id", "profile", "Profile",
                   "cluster", "Cluster", "segment", "Segment", "group", "Group"]


# ─────────────────────────────────────────────
# 2. LOAD CONTEXT MEMORY (from first agent)
# ─────────────────────────────────────────────

def load_context_memory(filepath: str = CONTEXT_MEMORY_FILE) -> str:
    """
    Loads the JSON memory produced by the first context agent
    and formats it as readable text for the prompt.
    """
    if not os.path.exists(filepath):
        raise FileNotFoundError(
            f"\n❌ '{filepath}' not found!\n"
            "   Run the context agent first to generate it."
        )
    with open(filepath, "r", encoding="utf-8") as f:
        memory = json.load(f)

    text = f"""
ROLE: {memory.get('ruolo', 'N/A')}
GENERAL CONTEXT: {memory.get('contesto_generale', 'N/A')}
BUSINESS CONTEXT: {memory.get('contesto_business', 'N/A')}
PURPOSE: {memory.get('scopo', 'N/A')}
FEATURES AND CONCEPTS: {', '.join(memory.get('features_e_concetti', []))}
""".strip()

    print(f"  [OK] Context memory loaded from '{filepath}'")
    return text


# ─────────────────────────────────────────────
# 3. READ CSV FILES
# ─────────────────────────────────────────────

def read_csv(filepath: str, sep: str = ";") -> pd.DataFrame:
    """
    Reads a CSV file with configurable separator.
    Tries UTF-8 first, then latin-1 as fallback.
    Returns empty DataFrame if file is not found.
    """
    if not os.path.exists(filepath):
        print(f"  [WARN] File not found: {filepath} — skipped.")
        return pd.DataFrame()

    for enc in ["utf-8", "latin-1"]:
        try:
            df = pd.read_csv(filepath, sep=sep, encoding=enc)
            print(f"  [OK] {filepath} — {df.shape[0]} rows, {df.shape[1]} cols")
            return df
        except Exception:
            continue

    print(f"  [ERR] Cannot read: {filepath}")
    return pd.DataFrame()


# ─────────────────────────────────────────────
# 4. READ SHAP JSON
# ─────────────────────────────────────────────

def read_shap_json(filepath: str) -> str:
    """
    Reads the SHAP importance JSON file and returns it
    as a formatted text string ready for the prompt.
    Returns 'Not available.' if the file is missing or invalid.
    """
    if not os.path.exists(filepath):
        print(f"  [WARN] SHAP file not found: {filepath} — skipped.")
        return "Not available."
    try:
        with open(filepath, "r", encoding="utf-8") as f:
            data = json.load(f)
        print(f"  [OK] {filepath} — SHAP data loaded.")
        # Convert JSON to readable text for the prompt
        return json.dumps(data, ensure_ascii=False, indent=2)
    except json.JSONDecodeError as e:
        print(f"  [ERR] Cannot parse {filepath}: {e}")
        return "Not available."


# ─────────────────────────────────────────────
# 5. PROFILE UTILITIES
# ─────────────────────────────────────────────

def find_profile_column(df: pd.DataFrame) -> str | None:
    """
    Returns the name of the profile column found in the DataFrame,
    or None if no known profile column exists.
    """
    for col in PROFILE_COLUMNS:
        if col in df.columns:
            return col
    return None


def extract_profiles(df_stats: pd.DataFrame) -> list:
    """
    Extracts the unique list of profile IDs from the stats DataFrame.
    Uses the first matching profile column from PROFILE_COLUMNS.
    Falls back to the first column if none is found.
    """
    col = find_profile_column(df_stats)
    if col:
        profiles = df_stats[col].unique().tolist()
        print(f"  [OK] Profile column found: '{col}' — {len(profiles)} unique profiles")
        return profiles

    fallback_col = df_stats.columns[0]
    print(f"  [WARN] No profile column found, using first column: '{fallback_col}'")
    return df_stats[fallback_col].unique().tolist()


def filter_by_profile(df: pd.DataFrame, profile_id: str) -> str:
    """
    Filters a DataFrame by profile_id and returns data as a clean
    'feature: value' list — much easier for the model to read than
    a flat tabular string with misaligned columns.
    If no profile column exists, returns the full DataFrame as-is
    (used for global data sources like odd ratios).
    """
    if df.empty:
        return "Not available."

    col = find_profile_column(df)
    if col:
        subset = df[df[col].astype(str) == str(profile_id)]
        if not subset.empty:
            # Convert each row to readable "feature: value" lines
            # This avoids misaligned column headers confusing the model
            lines = []
            for _, row in subset.iterrows():
                for colname, value in row.items():
                    if colname != col:  # skip the profile_id column itself
                        lines.append(f"  {colname}: {value}")
                lines.append("")  # blank line between rows
            return "\n".join(lines).strip()

    # No profile column — return full table (global data)
    return df.to_string(index=False)


# ─────────────────────────────────────────────
# 6. CALL OLLAMA
# ─────────────────────────────────────────────

def ask_ollama(messages: list) -> str:
    """
    Sends a message list to Ollama and returns the text response.
    Handles connection errors and timeouts gracefully.
    """
    payload = {
        "model": MODELLO,
        "messages": messages,
        "stream": False,
        "options": {
            "temperature": 0.45,   # balanced: creative but precise
            "num_predict": 1000   # max response length in tokens
        }
    }
    try:
        response = requests.post(OLLAMA_URL, json=payload, timeout=TIMEOUT)
        response.raise_for_status()
        return response.json()["message"]["content"]

    except requests.exceptions.ConnectionError:
        raise ConnectionError(
            "\n❌ Ollama is not running!\n"
            "   Start it with: ollama serve"
        )
    except requests.exceptions.Timeout:
        raise TimeoutError(
            f"\n❌ Timeout after {TIMEOUT}s.\n"
            "   Try reducing CSV data size or increase TIMEOUT."
        )


# ─────────────────────────────────────────────
# 7. FEEDBACK LOOP
# ─────────────────────────────────────────────

def feedback_loop(initial_response: str, history: list, label: str) -> str:
    """
    Shows the current model response and collects user feedback.
      - Free text  → refines the response (new call to Ollama)
      - "yes"      → confirms and exits the loop
    Returns the final confirmed response.
    """
    response = initial_response

    while True:
        print(f"\n{'─' * 60}")
        print(f"📋 {label}:\n")
        print(response)
        print(f"\n{'─' * 60}")
        print(f"\n✏️  Give feedback to refine, or type '{MAGIC_WORD}' to confirm:\n")

        user_input = input("You → ").strip()

        if user_input.lower() == MAGIC_WORD:
            print("✅ Confirmed!")
            return response

        if not user_input:
            print("  [INFO] Empty input, try again.")
            continue

        history.append({"role": "user", "content": user_input})
        print("\n⏳ Updating response...")
        response = ask_ollama(history)
        history.append({"role": "assistant", "content": response})


# ─────────────────────────────────────────────
# 8. BUILD PROFILE PROMPT
# ─────────────────────────────────────────────

def build_profile_prompt(
    context:   str,
    profile_id: str,
    stats:     str,
    shap:      str,
    features:  str,
    oddratio:  str,
    rules:     str
) -> str:
    """
    Builds the full prompt for a single profile.
    The profile_id is injected explicitly at the top so the model
    never has to guess or search for it in the tabular data.
    """
    return f"""
You are an expert data analyst and storyteller specialized in machine learning model output profiling.
You analyze and make actionable profiles based on ML model score.
You write compelling, data-driven, actionable profile descriptions for stakeholders.

---
## BUSINESS CONTEXT
{context}

---
## ⚠️ YOU ARE ANALYZING EXACTLY THIS PROFILE:
# Profile ID = {profile_id}
# This ID comes directly from the profile_stats file.
# Use it as-is. Do not change it, do not invent others.

---
## DATA FOR PROFILE ID: {profile_id}

### Statistical Summary (mean and mode per feature and Profile_ID)
{stats}

### SHAP Mean Values (average feature impact on model prediction)
{shap}

### Feature Importance (decision tree — top drivers for this profile)
{features}

### Odd Ratios (logistic regression — likelihood vs average population)
{oddratio}

### Assignment Rules (conditions that assign a customer to this profile)
{rules}

---
## YOUR TASK

Write a complete description of the profile with ID = {profile_id}.
Use ONLY the numbers and values listed above. Do NOT invent data.

Structure your answer as follows:

---
### 🎯 Profile Name & Tagline | ID: {profile_id}
- **Name**: A short memorable name (e.g. "The Loyal High-Spender")
- **Tagline**: One sentence capturing who they are

---
### 📖 Who Are They? (Storytelling)
3-5 sentences. Tell a vivid human story using the actual feature values above.
Ground every sentence in real data — e.g. "On average, they spend X per month and visit Y times."

---
### 📊 Key Data Insights
2-5 bullet points in this format:
- **[feature name]** = [value from data] → [what this means for the business]
Only use values explicitly present in the data above.

---
### ⚡ Actionable Recommendations
Provide 2-5 concrete, specific actions. Each must:
- Start with a verb (e.g. "Launch", "Target", "Reduce", "Increase")
- Reference specific data values to justify the action
- Be directly executable by a business team

---
### ⚠️ Watch Out For
List 1-2 risks or pitfalls specific to this profile based on the data.

---
Be specific. Be data-driven. Avoid generic statements like "this segment is important".
Every claim must trace back to a number or pattern in the data provided.
""".strip()


# ─────────────────────────────────────────────
# 9. ANALYZE A SINGLE PROFILE
# ─────────────────────────────────────────────

def analyze_profile(
    context:     str,
    profile_id:  str,
    shap_txt:    str,
    df_stats:    pd.DataFrame,
    df_features: pd.DataFrame,
    df_oddratio: pd.DataFrame,
    df_rules:    pd.DataFrame
) -> dict:
    """
    Generates the full description for a single profile:
      1. Filters each DataFrame for this profile_id
      2. Builds the prompt with all data + explicit profile_id
      3. Calls Ollama
      4. Opens feedback loop until user types "yes"
      5. Returns the confirmed description as a dict
    """
    print(f"\n  🔍 Analyzing profile: '{profile_id}'...")

    # Filter each data source for this specific profile
    stats_txt    = filter_by_profile(df_stats,    profile_id)
    features_txt = filter_by_profile(df_features, profile_id)
    oddratio_txt = filter_by_profile(df_oddratio, profile_id)
    rules_txt    = filter_by_profile(df_rules,    profile_id)

    # Build prompt — profile_id injected explicitly
    prompt = build_profile_prompt(
        context, profile_id,
        stats_txt, shap_txt, features_txt, oddratio_txt, rules_txt
    )

    # First model call
    history  = [{"role": "user", "content": prompt}]
    response = ask_ollama(history)
    history.append({"role": "assistant", "content": response})

    # Feedback loop — user can refine or confirm with "yes"
    final_description = feedback_loop(
        response, history,
        label=f"PROFILE DESCRIPTION — ID: {profile_id}"
    )

    return {
        "profile_id":  str(profile_id),
        "description": final_description
    }


# ─────────────────────────────────────────────
# 10. SAVE OUTPUT
# ─────────────────────────────────────────────

def save_output(results: list, filepath: str = OUTPUT_JSON):
    """Saves all profile descriptions to a structured JSON file."""
    output = {
        "total_profiles": len(results),
        "profiles": results
    }
    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(output, f, ensure_ascii=False, indent=2)
    print(f"\n✅ All profile descriptions saved to: {filepath}")


# ─────────────────────────────────────────────
# 11. MAIN FUNCTION
# ─────────────────────────────────────────────

def run_profile_agent(
    oddratio_csv:         str = "oddsratio.csv",
    profile_features_csv: str = "profile_features.csv",
    profile_stats_csv:    str = "profile_stats.csv",
    profile_rules_csv:    str = "profile_rules.csv",
    shap_importance:      str = "shap_importance.json",
    context_memory:       str = CONTEXT_MEMORY_FILE,
    sep:                  str = ";"
):
    """
    Main function of the profile analyst agent:
      1. Loads context memory from first agent
      2. Reads the profile CSVs + SHAP JSON
      3. Extracts unique profile IDs from profile_stats.csv
      4. For each profile: generates description → feedback loop → "yes"
      5. Saves everything to profile_descriptions.json
    """
    print("=" * 60)
    print("  PROFILE ANALYST AGENT — powered by Ollama phi3:mini")
    print("=" * 60)

    # STEP 1 — Load context from first agent
    print("\n📚 Loading business context memory...")
    context = load_context_memory(context_memory)

    # STEP 2 — Read profile CSVs
    print("\n📂 Loading profile CSV files...")
    df_oddratio  = read_csv(oddratio_csv,         sep)
    df_features  = read_csv(profile_features_csv, sep)
    df_stats     = read_csv(profile_stats_csv,    sep)
    df_rules     = read_csv(profile_rules_csv,    sep)

    # STEP 3 — Read SHAP JSON (global, not filtered per profile)
    print("\n📂 Loading SHAP JSON...")
    shap_txt = read_shap_json(shap_importance)

    # STEP 4 — Extract profile list from stats (most complete source)
    if df_stats.empty:
        raise ValueError(
            "❌ profile_stats.csv is empty or not found.\n"
            "   This file is required to extract the profile list."
        )
    profiles = extract_profiles(df_stats)
    print(f"\n🎯 Profiles to analyze: {profiles}")
    print(f"   Total: {len(profiles)}")

    # STEP 5 — Analyze each profile with feedback loop
    results = []
    for i, profile_id in enumerate(profiles, 1):
        print(f"\n{'█' * 60}")
        print(f"  PROFILE {i} of {len(profiles)}: '{profile_id}'")
        print(f"{'█' * 60}")

        result = analyze_profile(
            context, profile_id, shap_txt,
            df_stats, df_features, df_oddratio, df_rules
        )
        results.append(result)
        print(f"\n✅ Profile '{profile_id}' confirmed — moving to next.")

    # STEP 6 — Save all results
    save_output(results)

    print(f"\n🎉 Done! {len(results)} profiles analyzed.")
    print(f"   Output: {OUTPUT_JSON}")

    return results


In [ ]:
# ─────────────────────────────────────────────
# 12. ENTRY POINT
# ─────────────────────────────────────────────

if __name__ == "__main__":
    run_profile_agent(
        oddratio_csv         = "IF_analytical_output/oddsratio.csv",
        profile_features_csv = "IF_analytical_output/profile_features.csv",
        profile_stats_csv    = "IF_analytical_output/profile_stats.csv",
        profile_rules_csv    = "IF_analytical_output/profile_rules.csv",
        shap_importance      = "shap_importance.json",
        context_memory       = "context_memory.json",
        sep                  = ","
    )